# 7.23 — Image captioning

Captioning is a translation problem where the source language is visual evidence and the target language is a sentence spoken one token at a time.

A vision encoder supplies patch vectors before any word is generated. A language decoder cross-attends to those visual tokens, then a softmax head chooses the next word and incurs negative log-likelihood on the target token.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

## The concept, built once: patch memory to next word

Captioning uses image tokens as visual memory. A $4	imes4$ image with $2	imes2$ patches gives a $4	imes4$ memory, cross-attention mixes visual values, and next-token loss is

$$\operatorname{NLL}(y)=-\log p_y.$$

The same logits can make `dog` cheap and `runs` expensive.

In [ ]:
def softmax(x):
    x = np.asarray(x, dtype=float)
    shifted = x - x.max(axis=-1, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum(axis=-1, keepdims=True)

def patchify(image, patch_size):
    patches = []
    for r in range(0, image.shape[0], patch_size):
        for c in range(0, image.shape[1], patch_size):
            patches.append(image[r:r + patch_size, c:c + patch_size].ravel())
    return np.array(patches)

def caption_step(image):
    tokens = patchify(image, 2)
    positions = 0.03 * np.arange(tokens.shape[0])[:, None] * np.ones_like(tokens)
    memory = tokens + positions
    weights = np.array([0.731, 0.269])
    values = np.array([[2.0, 0.0], [0.0, 3.0]])
    visual_mix = weights @ values
    logits = np.array([0.1, 2.0, 0.5])
    probs = softmax(logits)
    nll_dog = float(-np.log(probs[1]))
    nll_runs = float(-np.log(probs[2]))
    return memory, visual_mix, probs, nll_dog, nll_runs

image = np.arange(16).reshape(4, 4) / 15
memory, visual_mix, probs, nll_dog, nll_runs = caption_step(image)
assert memory.shape == (4, 4)
assert np.allclose(np.round(memory[1] - patchify(image, 2)[1], 2), [0.03, 0.03, 0.03, 0.03])
assert np.allclose(np.round(visual_mix, 3), [1.462, 0.807])
assert np.allclose(np.round(probs, 3), [0.109, 0.728, 0.163])
assert round(nll_dog, 3) == 0.317
assert round(nll_runs, 2) == 1.82
print("visual memory", memory.shape)
print("visual mixture", np.round(visual_mix, 3))
print("next-token probabilities", np.round(probs, 3))
print("NLL dog", round(nll_dog, 3))
print("NLL runs", round(nll_runs, 2))

## Caption ladder: generated images with target words

This ladder grows from clean line captions to noisy multi-object scenes. The decoder predicts one content token from patch evidence, so token accuracy is the metric.

In [ ]:
VOCAB = ["line", "square", "circle", "noisy", "end"]


def draw_shape(kind, size, noise, seed):
    rng = np.random.default_rng(seed)
    img = rng.uniform(0, noise, size=(size, size))
    yy, xx = np.mgrid[0:size, 0:size]
    if kind == "line":
        img[:, size // 2] = 1
    if kind == "square":
        img[size // 4:3 * size // 4, size // 4:3 * size // 4] = 1
    if kind == "circle":
        circle = (xx - size / 2) ** 2 + (yy - size / 2) ** 2 <= (size / 4) ** 2
        img[circle] = 1
    return np.clip(img, 0, 1)

def make_caption_rung(name, kinds, size, noise, n):
    images = []
    targets = []
    for i in range(n):
        kind = kinds[i % len(kinds)]
        images.append(draw_shape(kind, size, noise, i))
        targets.append(VOCAB.index(kind))
    return name, np.array(images), np.array(targets)

caption_rungs = [
    make_caption_rung("D1 line vs square", ["line", "square"], 4, 0.02, 40),
    make_caption_rung("D2 add circles", ["line", "square", "circle"], 8, 0.05, 60),
    make_caption_rung("D3 noisy shapes", ["line", "square", "circle"], 8, 0.15, 90),
    make_caption_rung("D4 small evidence", ["line", "square", "circle"], 8, 0.25, 90),
    make_caption_rung("D5 cluttered captions", ["line", "square", "circle"], 8, 0.35, 120),
]

fig, axes = plt.subplots(1, 5, figsize=(11, 2.3))
for ax, (name, X, y) in zip(axes, caption_rungs):
    ax.imshow(X[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}\nnext={VOCAB[y[0]]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()

for name, X, y in caption_rungs:
    print(f"{name:24s} images={X.shape} target words={sorted(set(y.tolist()))}")

In [ ]:
def caption_features(img):
    tokens = patchify(img, 2)
    means = tokens.mean(axis=1)
    stds = tokens.std(axis=1)
    row_energy = img.mean(axis=1)
    col_energy = img.mean(axis=0)
    return np.concatenate([means, stds, row_energy, col_energy])

def token_accuracy(X, y):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    feats = np.array([caption_features(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te), clf

caption_scores = []
caption_models = []
for name, X, y in caption_rungs:
    score, clf = token_accuracy(X, y)
    caption_scores.append(score)
    caption_models.append(clf)
    print(f"{name:24s} token accuracy={score:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
for ax, (name, X, y), clf in zip(axes, caption_rungs, caption_models):
    features = caption_features(X[0]).reshape(2, -1)
    prediction = clf.predict([caption_features(X[0])])[0]
    ax.imshow(features, aspect="auto", cmap="viridis")
    ax.set_title(f"{name.split()[0]} pred {VOCAB[prediction]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()

plt.figure(figsize=(5, 3))
plt.plot(range(1, 6), caption_scores, marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0, 1.05)
plt.ylabel("next-token accuracy")
plt.title("Caption token prediction across the ladder")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on D5: tags without grounding or stop checks

A tag-only decoder can hallucinate common words. Cross-attention evidence and an explicit end token reduce unsupported captions and prevent infinite generation.

In [ ]:
name, X, y = caption_rungs[-1]
most_common = np.bincount(y).argmax()
tag_only = [most_common, VOCAB.index("noisy"), VOCAB.index("noisy")]
grounded = [caption_models[-1].predict([caption_features(X[0])])[0], VOCAB.index("end")]
unsupported_noisy = tag_only.count(VOCAB.index("noisy"))
stops = grounded[-1] == VOCAB.index("end")
print("tag-only caption", [VOCAB[i] for i in tag_only])
print("grounded caption", [VOCAB[i] for i in grounded])
print("unsupported noisy words", unsupported_noisy)
print("has stop token", stops)

## Evaluate it + Practice

- Metric: next-token accuracy and NLL; compare with the no-skill baseline, always predict the most common word.
- Cheap sanity check: D1 should choose the visible line or square.
- Ablation: remove patch features and keep only a class prior.
- Failure signals: common words appear without visual attention or no end token is emitted.

Practice:
1. Change one D1 number and predict the metric before running it.

In [ ]:
# Your experiment here

2. Add one harder case to the ladder and rerun the curve.

In [ ]:
# Your harder case here

3. Turn off the key fix in the pitfall cell and explain the change.

In [ ]:
# Your ablation here